# Notebook 2: MinHash Entity Resolution

Contractor names in federal procurement data contain abbreviation variants
(e.g. "LOCKHEED MARTIN CORPORATION" vs "LOCKHEED MARTIN CORP") that would
create duplicate nodes in our network graph. This notebook uses k-shingling
and MinHash LSH to identify and merge near-duplicate contractor names.

**Approach:** Character-level 3-shingles, 128 permutations, Jaccard similarity
threshold of 0.85. This conservative threshold merges only clear abbreviation
variants (e.g. "CORP" vs "CORPORATION") while deliberately preserving legally
distinct subsidiaries (e.g. "LOCKHEED MARTIN SERVICES LLC" vs "LOCKHEED MARTIN
INTEGRATED SYSTEMS LLC") as separate network nodes. This avoids over-merging
entities that are legally and operationally distinct, which would collapse
meaningful network structure.

**Input:** `contracts_cleaned.csv`  
**Output:** `contracts_resolved.csv`

In [36]:
import pandas as pd
import numpy as np
from datasketch import MinHash, MinHashLSH
import re
import roman

**Validate shape of cleaned data**

In [37]:
df = pd.read_csv('contracts_cleaned.csv', dtype={'naics_code': str})

print(f"Shape: {df.shape}")
print(f"Unique contractor names before resolution: {df['recipient_name'].nunique()}")

Shape: (73350, 6)
Unique contractor names before resolution: 24618


**Shingling & MinHashing functions:**

In [38]:
# Generate character-level k-shingles from a string
def get_shingles(text, k=3):
    text = re.sub(r'\s+', '', text)  # remove spaces only
    return set(text[i:i+k] for i in range(len(text) - k + 1))

# Create a MinHash signature from a set of shingles
def make_minhash(shingles, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in shingles:
        m.update(s.encode('utf-8'))
    return m

# Get unique names only: we only need one signature per unique string
unique_names = df['recipient_name'].unique()
print(f"Building MinHash signatures for {len(unique_names):,} unique names...")

signatures = {}
for name in unique_names:
    shingles = get_shingles(name)
    if len(shingles) > 0:
        signatures[name] = make_minhash(shingles)

print(f"Signatures built: {len(signatures):,}")

Building MinHash signatures for 24,618 unique names...
Signatures built: 24,618


**Build LSH index and find candidate pairs**

In [39]:
# Build LSH index
lsh = MinHashLSH(threshold=0.85, num_perm=128)

for name, sig in signatures.items():
    lsh.insert(name, sig)

# Query each name against the index to find similar pairs
print("Querying LSH index for similar name pairs...")

merge_pairs = []
seen = set()

for name, sig in signatures.items():
    results = lsh.query(sig)
    for match in results:
        if match != name:
            pair = tuple(sorted([name, match]))
            if pair not in seen:
                seen.add(pair)
                merge_pairs.append(pair)

print(f"Candidate pairs found: {len(merge_pairs)}")

Querying LSH index for similar name pairs...
Candidate pairs found: 655


In [40]:
US_STATES_AND_TERRITORIES = {
    'ALABAMA', 'ALASKA', 'ARIZONA', 'ARKANSAS', 'CALIFORNIA', 'COLORADO',
    'CONNECTICUT', 'DELAWARE', 'FLORIDA', 'GEORGIA', 'HAWAII', 'IDAHO',
    'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY', 'LOUISIANA',
    'MAINE', 'MARYLAND', 'MASSACHUSETTS', 'MICHIGAN', 'MINNESOTA',
    'MISSISSIPPI', 'MISSOURI', 'MONTANA', 'NEBRASKA', 'NEVADA',
    'NEW HAMPSHIRE', 'NEW JERSEY', 'NEW MEXICO', 'NEW YORK',
    'NORTH CAROLINA', 'NORTH DAKOTA', 'OHIO', 'OKLAHOMA', 'OREGON',
    'PENNSYLVANIA', 'RHODE ISLAND', 'SOUTH CAROLINA', 'SOUTH DAKOTA',
    'TENNESSEE', 'TEXAS', 'UTAH', 'VERMONT', 'VIRGINIA', 'WASHINGTON',
    'WEST VIRGINIA', 'WISCONSIN', 'WYOMING', 'GUAM', 'PUERTO RICO',
    'AMERICAN SAMOA'
}

def ends_with_geo_qualifier(name, other):
    """Returns True if name ends with a geographic qualifier
    the other name lacks — indicates geographic specialization,
    not a name variant."""
    for geo in US_STATES_AND_TERRITORIES:
        if name.endswith(geo) and not other.endswith(geo):
            return True
    return False

def first_word(name):
    tokens = name.split()
    return tokens[0] if tokens else ''

def extract_numbers(name):
    arabic = set(re.findall(r'\d+', name))
    roman_numerals = {'I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X'}
    tokens = set(name.split())
    return arabic | (tokens & roman_numerals)

filtered_pairs = []
for a, b in merge_pairs:
    if first_word(a) != first_word(b):
        continue
    if extract_numbers(a) != extract_numbers(b):
        continue
    if ends_with_geo_qualifier(a, b) or ends_with_geo_qualifier(b, a):
        continue
    filtered_pairs.append((a, b))

print(f"Pairs before filters: {len(merge_pairs)}")
print(f"After first-word filter: removed {len([p for p in merge_pairs if first_word(p[0]) != first_word(p[1])])}")
print(f"After numbers filter: removes numbered JV variants")
print(f"After geo filter: removes geographic specializations")
print(f"Final approved pairs: {len(filtered_pairs)}")

print("\n=== Filtered pairs ===\n")
for a, b in filtered_pairs:
    print(f"  '{a}'")
    print(f"  '{b}'")
    print()

Pairs before filters: 655
After first-word filter: removed 623
After numbers filter: removes numbered JV variants
After geo filter: removes geographic specializations
Final approved pairs: 11

=== Filtered pairs ===

  'HUNTSVILLE REHABILITATION FOUNDATION'
  'HUNTSVILLE REHABILITATION FOUNDATION INC'

  'MERIDIAN ENGINEERING CO'
  'MERIDIAN ENGINEERING INC'

  'WASTE MANAGEMENT OF VIRGINIA INC'
  'WASTE MANAGEMENT OF WEST VIRGINIA INC'

  'FEDERAL RESOURCES SUPPLY COMPANY'
  'FEDERAL RESOURCES SUPPLY COMPANY LLC'

  'UMASS MEMORIAL MEDICAL GROUP'
  'UMASS MEMORIAL MEDICAL GROUP INC'

  'L2EOCULUS JV'
  'L2EOCULUS JV LLC'

  'TYONEK SERVICES OVERHAUL FACILITY STENNIS LLC'
  'TYONEK SERVICES OVERHAUL FACILITYSTENNIS LLC'

  'VANDERBILT UNIVERSITY'
  'VANDERBILT UNIVERSITY THE'

  'SOUTHWESTERN BELL TELEPHONE COMPANY'
  'SOUTHWESTERN BELL TELEPHONE COMPANY LLC'

  'OMANG TECHNOLOGIES AND TRADING LLC'
  'OMANG TECHNOLOGIES AND TRADING LTD'

  'STERLING SHIPYARD LLC'
  'STERLING SHIPYARD L

**Canonical name mapping**

In [41]:
# Union-Find to handle transitive merges
parent = {name: name for name in unique_names}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        # Canonical name = longest string (most complete form)
        if len(px) >= len(py):
            parent[py] = px
        else:
            parent[px] = py

# Apply all filtered pairs
for a, b in filtered_pairs:
    union(a, b)

# Build name -> canonical name mapping
name_to_canonical = {name: find(name) for name in unique_names}

# Summary
num_merged = sum(1 for k, v in name_to_canonical.items() if k != v)
print(f"Names that will be merged into a canonical form: {num_merged}")
print(f"Unique entities before resolution: {len(unique_names):,}")
print(f"Unique entities after resolution: {len(set(name_to_canonical.values())):,}")

# Show some merge groups
from collections import defaultdict
groups = defaultdict(list)
for name, canonical in name_to_canonical.items():
    if name != canonical:
        groups[canonical].append(name)

print(f"\n=== Sample merge groups ===")
for canonical, variants in list(groups.items())[:10]:
    print(f"\n  Canonical: '{canonical}'")
    for v in variants:
        print(f"    <- '{v}'")

Names that will be merged into a canonical form: 11
Unique entities before resolution: 24,618
Unique entities after resolution: 24,607

=== Sample merge groups ===

  Canonical: 'HUNTSVILLE REHABILITATION FOUNDATION INC'
    <- 'HUNTSVILLE REHABILITATION FOUNDATION'

  Canonical: 'FEDERAL RESOURCES SUPPLY COMPANY LLC'
    <- 'FEDERAL RESOURCES SUPPLY COMPANY'

  Canonical: 'MERIDIAN ENGINEERING INC'
    <- 'MERIDIAN ENGINEERING CO'

  Canonical: 'WASTE MANAGEMENT OF WEST VIRGINIA INC'
    <- 'WASTE MANAGEMENT OF VIRGINIA INC'

  Canonical: 'UMASS MEMORIAL MEDICAL GROUP INC'
    <- 'UMASS MEMORIAL MEDICAL GROUP'

  Canonical: 'L2EOCULUS JV LLC'
    <- 'L2EOCULUS JV'

  Canonical: 'TYONEK SERVICES OVERHAUL FACILITY STENNIS LLC'
    <- 'TYONEK SERVICES OVERHAUL FACILITYSTENNIS LLC'

  Canonical: 'VANDERBILT UNIVERSITY THE'
    <- 'VANDERBILT UNIVERSITY'

  Canonical: 'STERLING SHIPYARD LLC'
    <- 'STERLING SHIPYARD LP'

  Canonical: 'SOUTHWESTERN BELL TELEPHONE COMPANY LLC'
    <- 'SOUTH

**Apply mapping to df and save**

In [42]:
# Apply canonical name mapping to full dataframe
df['recipient_name_original'] = df['recipient_name']
df['recipient_name'] = df['recipient_name'].map(name_to_canonical)

print(f"Unique contractor names before resolution: {df['recipient_name_original'].nunique():,}")
print(f"Unique contractor names after resolution: {df['recipient_name'].nunique():,}")
print(f"Null recipient_names after mapping: {df['recipient_name'].isna().sum()}")

# Save
df.to_csv('contracts_resolved.csv', index=False)
print("\nSaved contracts_resolved.csv")

Unique contractor names before resolution: 24,618
Unique contractor names after resolution: 24,607
Null recipient_names after mapping: 0

Saved contracts_resolved.csv
